In [1077]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re

In [1078]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')

In [1079]:
device

'cuda'

In [1080]:
train_dataset = pd.read_csv("train.csv", encoding="latin1")
test_dataset = pd.read_csv("test.csv", encoding="latin1")

In [1081]:
train_dataset.columns

Index(['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'],
      dtype='object')

In [1082]:
train_dataset.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [1083]:
test_dataset.columns

Index(['textID', 'text', 'sentiment', 'Time of Tweet', 'Age of User',
       'Country', 'Population -2020', 'Land Area (Kmï¿½)',
       'Density (P/Kmï¿½)'],
      dtype='object')

In [1084]:
test_dataset.head()

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Kmï¿½),Density (P/Kmï¿½)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797,27400.0,105
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044,2381740.0,18
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265,470.0,164
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272,1246700.0,26


In [1085]:
train_dataset = train_dataset.loc[:,["text", "sentiment"]]

In [1086]:
test_dataset = test_dataset.loc[:, ["text", "sentiment"]]

In [1087]:
train_dataset.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [1088]:
test_dataset.head()

,text,sentiment
0,Last session of the day http://twitpic.com/67ezh,neutral
1,Shanghai is also really exciting (precisely -...,positive
2,"Recession hit Veronique Branquinho, she has to...",negative
3,happy bday!,positive
4,http://twitpic.com/4w75p - I like it!!,positive


In [1089]:
train_dataset.shape

(27481, 2)

In [1090]:
def tokenizer(text):
    text = str(text)
    lower_case_text = text.lower()
    # removes the characters that are not letter or digit or space with ""
    text = re.sub(r"[^\w\d\s]", "", lower_case_text)
    return text.split()

In [1091]:
tokenizer("Hello MY NaMe is AbhIHsheK TimMsInA")

['hello', 'my', 'name', 'is', 'abhihshek', 'timmsina']

In [1092]:
tokenizer(train_dataset.iloc[0,0])

['id', 'have', 'responded', 'if', 'i', 'were', 'going']

In [1093]:
answers = {}
choices = train_dataset.iloc[:, 1].unique()
print(choices)
for i in choices:
    answers[i] = len(answers)
answers

['neutral' 'negative' 'positive']


{'neutral': 0, 'negative': 1, 'positive': 2}

In [1094]:
# voccabulary for stroing the tokens
voccab = {
    '<UNK>' : 0
}

# for building the voccabulary for training as it is done in timestep, each token means one timestep
def build_voccab(dataset):
    for text_index in range(dataset.shape[0]):
        text = dataset.iloc[text_index,0]
        # tokenize
        tokens = tokenizer(text)
        # add in voccab
        for token in tokens:
            if token not in voccab:
                voccab[token] = len(voccab)
    print("voccab created ✅")

In [1095]:
build_voccab(train_dataset)

voccab created ✅


In [1096]:
len(voccab)

29584

In [1097]:
def text_to_indices(text, voccab):
    tokens = tokenizer(text)
    indices = []
    if len(tokens) == 0:
        indices.append(voccab['<UNK>'])
        return torch.tensor(indices)
    for token in tokens:
        if token in voccab:
            indices.append(voccab[token])
        else:
            indices.append(voccab["<UNK>"])
    return torch.tensor(indices)

In [1098]:
indices = text_to_indices("i am dsd", voccab)
indices

tensor([  5, 112,   0])

In [1099]:
len(voccab)

29584

In [1100]:
# location where the data are stored
class CustomDataSet(Dataset):
    def __init__(self, df, voccab):
        self.df = df
        self.voccab = voccab
        print(self.df.shape)
        print(len(self.voccab))

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        indices = text_to_indices(self.df.iloc[index, 0], self.voccab)
        answer = torch.tensor(answers[self.df.iloc[index, 1]])
        return indices, answer

In [1101]:
train_data = CustomDataSet(train_dataset, voccab)

(27481, 2)
29584


In [1102]:
train_data.__getitem__(0)

(tensor([1, 2, 3, 4, 5, 6, 7]), tensor(0))

In [1103]:
train_dataloader = DataLoader(dataset = train_data, batch_size = 64, shuffle = True)

In [1104]:
len(train_dataloader)

430

In [1108]:
class RNN(nn.Module):
    def __init__(self, size_of_voccab, size_of_embedding):

        super().__init__()
        
        # embedding so that the model can understand the tokens as the integer is nothing but id
        # we need a embedding that explains the meaning of the token to the rnn
        # as the model learns, the embedding also improves
        self.embedding = nn.Embedding(num_embeddings= size_of_voccab, embedding_dim= size_of_embedding)
        # RNN as the sequential model, that process the embedding of each token sequentially and understands/learns/summarizes the tokens one timestep at a time
        # it uses the backward loop to use previous hidden state in the hidden layer of next time step as context/memroy to process the nextt token
        # with such mechanism it keeps summarizing till the end and
        # it gives 64 output by taking embedding of each token
        self.rnn = nn.RNN(size_of_embedding, 64)
        # linear is used to combine the output into 3 neurons(positive, negative and neutral)
        self.linear = nn.Linear(64, 3)

    # Forward Propagation
    def forward(self, tokens_indices):
        embeddings = self.embedding(tokens_indices)
        rnn_output = self.rnn(embeddings)
        linear = self.linear(rnn_output[1])
        return linear

In [1109]:
rnn_model = RNN(len(voccab), 50)
rnn_model.to(device=device)

RNN(
  (embedding): Embedding(29584, 50)
  (rnn): RNN(50, 64)
  (linear): Linear(in_features=64, out_features=3, bias=True)
)

In [1110]:
train_data[0]

(tensor([1, 2, 3, 4, 5, 6, 7]), tensor(0))

In [1111]:
for i in rnn_model.named_parameters():
    print(i)

('embedding.weight', Parameter containing:
tensor([[ 1.0692,  1.0807,  1.2554,  ..., -1.3476,  0.6221, -1.8506],
        [ 1.0206,  0.4352, -0.4132,  ..., -0.6703, -0.7866,  0.3732],
        [ 0.3996, -0.2840, -0.2731,  ...,  0.7837,  0.2047, -1.9735],
        ...,
        [ 0.6666, -1.6364, -0.7351,  ..., -1.4142,  2.1852,  0.7464],
        [ 0.3243,  0.4817, -0.3267,  ...,  0.6472,  0.7841, -0.1572],
        [-0.5624,  0.2795, -1.2253,  ..., -0.0817, -0.3387, -0.1879]],
       device='cuda:0', requires_grad=True))
('rnn.weight_ih_l0', Parameter containing:
tensor([[ 0.1070, -0.0211,  0.0659,  ..., -0.1209,  0.0525, -0.0922],
        [-0.1149,  0.1098, -0.0940,  ..., -0.0060,  0.0349,  0.0404],
        [ 0.1049, -0.0551, -0.0013,  ...,  0.0801,  0.0340, -0.0511],
        ...,
        [ 0.0366, -0.0597,  0.1047,  ...,  0.1198, -0.0913,  0.0551],
        [-0.0070, -0.0154,  0.0970,  ...,  0.0124,  0.0737, -0.0457],
        [ 0.0641,  0.0257,  0.0150,  ...,  0.1066,  0.0745, -0.1038]],
 

In [1112]:
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.Adam(rnn_model.parameters(), lr = 0.001)

In [1113]:
epochs = 25

In [ ]:
# training the RNN model:
for epoch in range(epochs):
    tota_loss = 0
    for que, ans in train_dataloader:
        que = que.reshape(que.shape[1]).to(device)
        ans = ans.to(device)
        y_pred = rnn_model(que)
        optim.zero_grad()
        loss = loss_fn(y_pred, ans)
        loss.backward()
        optim.step()
        tota_loss = tota_loss + loss.item()
    print("Epoch :", epoch, "===============  Loss :", tota_loss/len(train_dataloader))

In [1114]:
tokens = tokenizer(test_dataset.iloc[4, 0])
tokens

['httptwitpiccom4w75p', 'i', 'like', 'it']

In [1115]:
index = text_to_indices(tokens, voccab).to(device=device)
index

tensor([  0,   5,  88, 133], device='cuda:0')

In [1116]:
rnn_model(index)

tensor([[-0.2737,  0.2582, -0.2757]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

In [1117]:
test_dataset.shape

(3534, 2)

## testing using the test

In [1118]:
test_data = CustomDataSet(test_dataset, voccab)
test_dataloader = DataLoader(test_data, batch_size= 1, shuffle= False)

(3534, 2)
29584


In [1119]:
test_data.__len__()

3534

In [1120]:
test_data[1]

(tensor([    0,    19,   374,    87,  1476, 13469,     0, 16109,   454,  5523,
            14,  5780, 13200,     0]),
 tensor(2))

In [1121]:
rnn_model.eval()

RNN(
  (embedding): Embedding(29584, 50)
  (rnn): RNN(50, 64)
  (linear): Linear(in_features=64, out_features=3, bias=True)
)

In [1122]:
correct = 0
for question, answer in test_dataloader:
    question = question.reshape(question.shape[1]).to(device)
    answer = answer.to(device)
    y_pred = rnn_model(question)
    if answer == torch.max(y_pred, dim= 1)[1]:
        correct = correct + 1
    print("real :", answer)
    print(y_pred)
    print("pred :", torch.max(y_pred, dim = 1)[1])
    print("="*100)
print("Accuracy :", correct/len(test_dataloader))

real : tensor([0], device='cuda:0')
tensor([[-0.2598, -0.0441, -0.0953]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
pred : tensor([1], device='cuda:0')
real : tensor([2], device='cuda:0')
tensor([[-0.5137,  0.0172, -0.1615]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
pred : tensor([1], device='cuda:0')
real : tensor([1], device='cuda:0')
tensor([[ 0.0304, -0.2036,  0.1550]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
pred : tensor([2], device='cuda:0')
real : tensor([2], device='cuda:0')
tensor([[0.0250, 0.1772, 0.2616]], device='cuda:0', grad_fn=<AddmmBackward0>)
pred : tensor([2], device='cuda:0')
real : tensor([2], device='cuda:0')
tensor([[-0.2737,  0.2582, -0.2757]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
pred : tensor([1], device='cuda:0')
real : tensor([2], device='cuda:0')
tensor([[ 0.5595, -0.2055,  0.1967]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
pred : tensor([0], device='cuda:0')
real : tensor([1], device='cuda:0')
tensor([[-0.05